In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/mohamed2ramadan1/lora-bart-sumarrizer/transformers/default/1/adapter_model.safetensors
/kaggle/input/models/mohamed2ramadan1/lora-bart-sumarrizer/transformers/default/1/adapter_config.json
/kaggle/input/models/mohamed2ramadan1/lora-bart-sumarrizer/transformers/default/1/README.md
/kaggle/input/models/mohamed2ramadan1/lora-bart-sumarrizer/transformers/default/1/tokenizer.json
/kaggle/input/models/mohamed2ramadan1/lora-bart-sumarrizer/transformers/default/1/tokenizer_config.json


In [2]:
!pip install transformers[torch] datasets evaluate rouge_score peft -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [4]:
from transformers import BartForConditionalGeneration, BartTokenizer
from peft import PeftModel
import torch
from datasets import load_dataset
from transformers import (
    BartForConditionalGeneration, 
    BartTokenizer, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, TaskType
model_path = "/kaggle/input/models/mohamed2ramadan1/lora-bart-sumarrizer/transformers/default/1" # The folder where you saved the model

# 1. Load Tokenizer
tokenizer = BartTokenizer.from_pretrained(model_path)

# 2. Load the BASE model again
base_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

# 3. Load your SAVED weights onto the base model
# is_trainable=True is the key here!
model = PeftModel.from_pretrained(base_model, model_path, is_trainable=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [5]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
train_data = dataset["train"].shuffle(seed=42).select(range(59000))
val_data = dataset["validation"].select(range(2000)) 

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [6]:
def preprocess_function(examples):
    # 1. Tokenize the input articles (Encoder side)
    inputs = [doc for doc in examples["article"]]
    model_inputs = tokenizer(
        inputs, 
        max_length=1024, 
        truncation=True, 
        padding="max_length"
    )
    
    # 2. Tokenize the target summaries (Decoder side)
    # The 'text_target' parameter replaces the old 'as_target_tokenizer' logic
    labels = tokenizer(
        text_target=examples["highlights"], 
        max_length=128, 
        truncation=True, 
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)

Map:   0%|          | 0/59000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [8]:

# Setup your 100k dataset again (or a new subset)
# ... [Assuming you have tokenized_train and tokenized_val ready] ...

# 4. Refined Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-summarizer-continued",
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=5e-5,           # Lower learning rate for fine-tuning the fine-tune
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=4,           # One more pass
    fp16=True,
    save_total_limit=2,
    predict_with_generate=True,
    report_to="none"
)

# 5. Re-initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

# 6. Start training again
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,29.656639,7.206481
2000,29.658611,7.208880
3000,29.635383,7.205399


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=3688, training_loss=29.65585360247763, metrics={'train_runtime': 19087.6619, 'train_samples_per_second': 12.364, 'train_steps_per_second': 0.193, 'total_flos': 1.45180657188864e+17, 'train_loss': 29.65585360247763, 'epoch': 4.0})

In [9]:
# 1. حفظ الموديل والـ Tokenizer نهائياً بعد انتهاء التدريب
final_save_path = "./final_summarizer_model"
trainer.save_model(final_save_path)
tokenizer.save_pretrained(final_save_path)

# 2. ضغط الموديل في ملف Zip عشان تقدر تنزله بسهولة
import shutil
shutil.make_archive("summarizer_model_v2", 'zip', final_save_path)

print("✅ التدريب انتهى وتم حفظ الموديل وضغطه بنجاح!")

✅ التدريب انتهى وتم حفظ الموديل وضغطه بنجاح!
